In [3]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# ==================== CẤU HÌNH ====================
INPUT_FOLDER = "FinancialIndicator"
OUTPUT_FOLDER = "Data"

# Tạo thư mục output nếu chưa có
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ==================== DANH SÁCH CHỈ TIÊU MONG MUỐN (đã khớp tên) ====================
desired_indicators = [
    "Doanh thu thuần", "Tăng trưởng doanh thu",
    "Lợi nhuận gộp", "Tăng trưởng lợi nhuận gộp",
    "Lợi nhuận thuần từ HĐKD", "Tăng trưởng lợi nhuận thuần",
    "Lợi nhuận trước thuế", "Tăng trưởng lợi nhuận trước thuế",
    "Lợi nhuận sau thuế của Cổ đông công ty mẹ", "Tăng trưởng lợi nhuận sau thuế",

    "Tổng tài sản", "Tăng trưởng Tổng tài sản",
    "Tài sản ngắn hạn", "Tăng trưởng tài sản ngắn hạn",
    "Tài sản dài hạn", "Tăng trưởng tài sản dài hạn",
    "Nợ ngắn hạn", "Tăng trưởng nợ ngắn hạn",
    "Vay và nợ thuê tài chính ngắn hạn", "Tăng trưởng vay và nợ thuê tài chính ngắn hạn",
    "Nợ dài hạn", "Tăng trưởng Nợ dài hạn",
    "Vốn chủ sở hữu", "Tăng trưởng vốn chủ sở hữu",
    "Vốn góp cổ phần", "Tăng trưởng vốn góp cổ phần",

    "P/E", "P/B", "EV/EBITDA",
    "EPS (VNĐ/CP)", "Tăng trưởng EPS", "Giá trị sổ sách (VNĐ/CP)",

    "Biên lợi nhuận gộp", "Biên EBIT", "Biên EBITDA", "Biên lợi nhuận ròng",
    "ROE LTM", "ROA LTM",

    "Vòng quay tài sản (vòng)", "Hiệu suất sử dụng tài sản cố định",
    "Số ngày thu tiền khách hàng (ngày)", "Số ngày xử lý hàng tồn kho (ngày)",
    "Số ngày phải trả nhà cung cấp (ngày)", "Vòng quay tiền mặt (ngày)",

    "Nợ phải trả / Vốn chủ sở hữu",
    "Vay và nợ thuê tài chính ngắn hạn, dài hạn / Vốn chủ sở hữu",
    "Nợ vay ròng / Vốn chủ sở hữu",
    "Tổng tài sản / Vốn chủ sở hữu",

    "Khả năng thanh toán tổng quát", "Khả năng thanh toán nhanh",
    "Khả năng thanh toán tức thời", "Khả năng thanh toán lãi vay",

    "Vốn hóa (Tỷ đồng)", "Số lượng cổ phiếu lưu hành (Triệu CP)"
]

# Các chỉ tiêu tuyệt đối cần chia cho 1 tỷ để ra đơn vị Tỷ đồng
absolute_indicators = [
    "Doanh thu thuần", "Lợi nhuận gộp", "Lợi nhuận thuần từ HĐKD",
    "Lợi nhuận trước thuế", "Lợi nhuận sau thuế của Cổ đông công ty mẹ",
    "Tổng tài sản", "Tài sản ngắn hạn", "Tài sản dài hạn",
    "Nợ ngắn hạn", "Vay và nợ thuê tài chính ngắn hạn", "Nợ dài hạn",
    "Vốn chủ sở hữu", "Vốn góp cổ phần", "Vốn hóa (Tỷ đồng)",
    "Số lượng cổ phiếu lưu hành (Triệu CP)"
]

# ==================== HÀM XỬ LÝ MỘT FILE ====================
def process_financial_file(file_path):
    company_name = Path(file_path).stem  # Lấy tên file không có đuôi
    
    print(f"\nĐang xử lý: {company_name} ...")
    
    try:
        # Đọc file, header bắt đầu từ dòng thứ 7 (header=6)
        df = pd.read_excel(file_path, header=6)
        
        # Loại bỏ footer (dòng toàn NaN)
        df = df.dropna(how='all', subset=df.columns[1:]).reset_index(drop=True)
        
        # Rename cột đầu tiên
        if 'CHỈ TIÊU' in df.columns:
            df = df.rename(columns={'CHỈ TIÊU': 'Chi_tieu'})
        
        # Giữ lại các dòng có dữ liệu số
        df_clean = df[df.iloc[:, 1:].notna().any(axis=1)].copy().reset_index(drop=True)
        
        # Làm sạch tên chỉ tiêu
        df_clean['Chi_tieu'] = df_clean['Chi_tieu'].astype(str).str.strip()
        
        # ==================== CHUYỂN SANG LONG FORMAT ====================
        value_vars = [col for col in df_clean.columns if isinstance(col, str) and col.startswith('Q')]
        
        if not value_vars:
            print(f"  ⚠ Không tìm thấy cột quý nào trong file {company_name}")
            return None
        
        df_long = pd.melt(df_clean,
                          id_vars=['Chi_tieu'],
                          value_vars=value_vars,
                          var_name='Quarter',
                          value_name='Value')
        
        # Tách Year và Quarter
        df_long['Year'] = df_long['Quarter'].str.extract(r'Q\d/(\d{4})').astype(int)
        df_long['Q'] = df_long['Quarter'].str.extract(r'(Q\d)/').astype(str)
        
        # Tạo cột datetime để sắp xếp
        df_long['Quarter_dt'] = pd.to_datetime(
            df_long['Quarter'].str.replace('Q', '').str.replace('/', '-'),
            format='%m-%Y', 
            errors='coerce'
        )
        
        df_long = df_long.sort_values('Quarter_dt', ascending=False).reset_index(drop=True)
        
        # ==================== CHUYỂN ĐƠN VỊ ====================
        mask_absolute = df_long['Chi_tieu'].isin(absolute_indicators)
        df_long.loc[mask_absolute, 'Value'] = df_long.loc[mask_absolute, 'Value'] / 1_000_000_000
        df_long['Value'] = df_long['Value'].round(2)
        
        # ==================== LỌC CHỈ TIÊU MONG MUỐN ====================
        df_filtered = df_long[df_long['Chi_tieu'].isin(desired_indicators)].copy().reset_index(drop=True)
        
        # ==================== KIỂM TRA CHỈ TIÊU KHÔNG KHỚP ====================
        missing = [ind for ind in desired_indicators if ind not in df_filtered['Chi_tieu'].unique()]
        if missing:
            print(f"  ⚠ Thiếu {len(missing)} chỉ tiêu: {missing[:10]}{'...' if len(missing)>10 else ''}")
        
        # ==================== LƯU FILE ====================
        output_path = os.path.join(OUTPUT_FOLDER, f"{company_name}_Data.xlsx")
        df_filtered.drop(columns=['Quarter', 'Quarter_dt'], errors='ignore').to_excel(output_path, index=False)
        
        print(f"  ✓ Hoàn thành {company_name} | {len(df_filtered['Chi_tieu'].unique())} chỉ tiêu | {df_filtered.shape[0]} dòng")
        
        return df_filtered
        
    except Exception as e:
        print(f"  ❌ Lỗi khi xử lý {company_name}: {e}")
        return None


# ==================== CHẠY XỬ LÝ TẤT CẢ FILE ====================
if __name__ == "__main__":
    # Lấy tất cả file .xlsx trong thư mục input
    excel_files = list(Path(INPUT_FOLDER).glob("*.xlsx"))
    
    print(f"Tìm thấy {len(excel_files)} file Excel trong thư mục {INPUT_FOLDER}/")
    
    if not excel_files:
        print("Không tìm thấy file nào!")
    else:
        for file_path in excel_files:
            process_financial_file(file_path)
        
        print("\n" + "="*60)
        print("HOÀN TẤT XỬ LÝ TẤT CẢ CÁC FILE!")
        print(f"Kết quả được lưu trong thư mục: ./{OUTPUT_FOLDER}/")
        print("="*60)

Tìm thấy 7 file Excel trong thư mục FinancialIndicator/

Đang xử lý: FPT ...


c:\Users\ThaiTu\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✓ Hoàn thành FPT | 54 chỉ tiêu | 2160 dòng

Đang xử lý: GAS ...


c:\Users\ThaiTu\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✓ Hoàn thành GAS | 54 chỉ tiêu | 2160 dòng

Đang xử lý: HPG ...


...

c:\Users\ThaiTu\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\ThaiTu\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✓ Hoàn thành MSN | 54 chỉ tiêu | 2160 dòng

Đang xử lý: MWG ...


c:\Users\ThaiTu\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✓ Hoàn thành MWG | 54 chỉ tiêu | 2160 dòng

Đang xử lý: VNM ...
  ✓ Hoàn thành VNM | 54 chỉ tiêu | 2160 dòng

HOÀN TẤT XỬ LÝ TẤT CẢ CÁC FILE!
Kết quả được lưu trong thư mục: ./Data/


c:\Users\ThaiTu\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [5]:
import pandas as pd
import os
from pathlib import Path

# ==================== CẤU HÌNH ====================
INPUT_FOLDER = "Data"
OUTPUT_FOLDER = "Processed"

# Tạo thư mục output nếu chưa tồn tại
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ==================== DANH SÁCH CHỈ TIÊU CẦN LẤY (CHÍNH XÁC) ====================
features = [
    "Tài sản ngắn hạn",
    "Nợ ngắn hạn",
    "Lợi nhuận thuần từ HĐKD",
    "Nợ dài hạn",
    "Vốn chủ sở hữu",
    "Tổng tài sản",
    "Tài sản dài hạn",
    "Doanh thu thuần",
    "Vòng quay tài sản (vòng)",
    "Số ngày xử lý hàng tồn kho (ngày)",
    "ROE LTM",
    "ROA LTM",
    "Lợi nhuận sau thuế của Cổ đông công ty mẹ",
    "Biên lợi nhuận gộp",
    "Biên EBIT",
    "Biên EBITDA",
    "Biên lợi nhuận ròng",
    "Nợ phải trả / Vốn chủ sở hữu",
    "P/E",
    "P/B",
    "Khả năng thanh toán tức thời"
]

# ==================== HÀM XỬ LÝ MỘT FILE ====================
def process_company_file(file_path):
    company_code = Path(file_path).stem.replace("_Data", "")  # Ví dụ: FPT, HPG, VNM...
    
    print(f"\nĐang xử lý: {company_code} ...")
    
    try:
        # Đọc file
        df = pd.read_excel(file_path)
        
        # Lọc chỉ tiêu cần thiết
        df_features = df[df['Chi_tieu'].isin(features)].copy()
        
        # Kiểm tra chỉ tiêu thiếu
        missing = [f for f in features if f not in df_features['Chi_tieu'].unique()]
        if missing:
            print(f"  ⚠️  Thiếu {len(missing)} chỉ tiêu: {missing}")
        
        print(f"  ✓ Tìm thấy {df_features['Chi_tieu'].nunique()}/{len(features)} chỉ tiêu")
        
        # ==================== CHUYỂN SANG WIDE FORMAT ====================
        # Đảm bảo có cột Year và Q
        if 'Year' not in df_features.columns or 'Q' not in df_features.columns:
            print(f"  ❌ File {company_code} thiếu cột Year hoặc Q")
            return
        
        df_wide = df_features.pivot(index=['Year', 'Q'], 
                                    columns='Chi_tieu', 
                                    values='Value').reset_index()
        
        # Sắp xếp theo thời gian (quý mới nhất lên trên)
        df_wide = df_wide.sort_values(['Year', 'Q'], ascending=[False, False]).reset_index(drop=True)
        
        # Xóa tên cột đa cấp
        df_wide.columns.name = None
        
        # ==================== LƯU FILE ====================
        long_path = os.path.join(OUTPUT_FOLDER, f"{company_code}_Selected_Features_Long.xlsx")
        wide_path = os.path.join(OUTPUT_FOLDER, f"{company_code}_Selected_Features_Wide.xlsx")
        
        df_features.to_excel(long_path, index=False)
        df_wide.to_excel(wide_path, index=False)
        
        print(f"  ✅ Hoàn thành {company_code}")
        print(f"     • Long : {df_features.shape[0]} dòng")
        print(f"     • Wide : {df_wide.shape[0]} quý x {df_wide.shape[1]-2} chỉ tiêu")
        
        return df_wide
        
    except Exception as e:
        print(f"  ❌ Lỗi khi xử lý {company_code}: {e}")
        return None


# ==================== CHẠY XỬ LÝ TẤT CẢ FILE ====================
if __name__ == "__main__":
    # Lấy tất cả file *_Data.xlsx trong thư mục Data
    data_files = list(Path(INPUT_FOLDER).glob("*_Data.xlsx"))
    
    print(f"Tìm thấy {len(data_files)} file dữ liệu trong thư mục '{INPUT_FOLDER}/'\n")
    
    if not data_files:
        print("Không tìm thấy file nào có đuôi _Data.xlsx")
    else:
        processed_count = 0
        for file_path in data_files:
            result = process_company_file(file_path)
            if result is not None:
                processed_count += 1
        
        print("\n" + "="*70)
        print(f"HOÀN TẤT! Đã xử lý {processed_count}/{len(data_files)} công ty")
        print(f"Kết quả được lưu trong thư mục: ./{OUTPUT_FOLDER}/")
        print("="*70)
        
        # Hiển thị danh sách file vừa tạo
        print("\nCác file Wide format đã tạo:")
        for f in sorted(Path(OUTPUT_FOLDER).glob("*_Wide.xlsx")):
            print("   •", f.name)

Tìm thấy 7 file dữ liệu trong thư mục 'Data/'


Đang xử lý: FPT ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành FPT
     • Long : 840 dòng
     • Wide : 40 quý x 21 chỉ tiêu

Đang xử lý: GAS ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành GAS
     • Long : 840 dòng
     • Wide : 40 quý x 21 chỉ tiêu

Đang xử lý: HPG ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành HPG
     • Long : 840 dòng
     • Wide : 40 quý x 21 chỉ tiêu

Đang xử lý: IMP ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành IMP
     • Long : 420 dòng
     • Wide : 20 quý x 21 chỉ tiêu

Đang xử lý: MSN ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành MSN
     • Long : 840 dòng
     • Wide : 40 quý x 21 chỉ tiêu

Đang xử lý: MWG ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành MWG
     • Long : 840 dòng
     • Wide : 40 quý x 21 chỉ tiêu

Đang xử lý: VNM ...
  ✓ Tìm thấy 21/21 chỉ tiêu
  ✅ Hoàn thành VNM
     • Long : 840 dòng
     • Wide : 40 quý x 21 chỉ tiêu

HOÀN TẤT! Đã xử lý 7/7 công ty
Kết quả được lưu trong thư mục: ./Proce